In [1]:
!pip install requests beautifulsoup4 pandas

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import csv
import time
from urllib.parse import urljoin

BASE_URL = "https://books.toscrape.com/"

headers = {"User-Agent": "Mozilla/5.0"}

def get_soup(url):
    response = requests.get(
        url, headers=headers, timeout=30
    )
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

def get_categories():
    soup = get_soup(BASE_URL)
    categories = {}

    for link in soup.select(
        "div.side_categories ul li ul li a"
    ):
        name = link.get_text(strip=True)
        url = urljoin(BASE_URL, link["href"])
        categories[name] = url

    return categories

def scrape_category(category, url):
    books = []
    next_page = url

    while next_page:
        soup = get_soup(next_page)

        for item in soup.select("article.product_pod"):
            title = item.h3.a["title"]

            price_text = item.select_one(
                ".price_color"
            ).get_text(strip=True)

            price_gbp = float(
                price_text.replace("£", "").replace("Â", "")
            )

            rating_word = item.select_one(
                "p.star-rating"
            )["class"][1]

            rating_map = {
                "One": 1, "Two": 2, "Three": 3,
                "Four": 4, "Five": 5
            }

            availability_text = item.select_one(
                ".availability"
            ).get_text(" ", strip=True)

            books.append({
                "title": title,
                "category": category,
                "price_gbp": price_gbp,
                "rating": rating_map[rating_word],
                "availability": "In stock" in availability_text,
                "book_url": urljoin(
                    next_page, item.h3.a["href"]
                )
            })

        next_link = soup.select_one("li.next a")

        next_page = (
            urljoin(next_page, next_link["href"])
            if next_link else None
        )

        time.sleep(0.2)

    return books

categories = get_categories()
selected_categories = list(categories.items())[:3]

all_books = []

for category, url in selected_categories:
    all_books.extend(scrape_category(category, url))

df = pd.DataFrame(all_books)
df.to_csv("books.csv", index=False)

print("Categories scraped:", len(selected_categories))
print("Total books scraped:", len(df))
display(df.head())

Categories scraped: 3
Total books scraped: 69


,title,category,price_gbp,rating,availability,book_url
0,It's Only the Himalayas,Travel,45.17,2,True,https://books.toscrape.com/catalogue/its-only-...
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Travel,49.43,4,True,https://books.toscrape.com/catalogue/full-moon...
2,See America: A Celebration of Our National Par...,Travel,48.87,3,True,https://books.toscrape.com/catalogue/see-ameri...
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,2,True,https://books.toscrape.com/catalogue/vagabondi...
4,Under the Tuscan Sun,Travel,37.33,3,True,https://books.toscrape.com/catalogue/under-the...


In [4]:
GBP_TO_INR = 105.50

df = pd.read_csv("books.csv")

df.drop_duplicates(
    subset=["title", "category"],
    inplace=True
)

df["title"] = df["title"].astype(str).str.strip()
df["price_gbp"] = pd.to_numeric(
    df["price_gbp"], errors="coerce"
)
df["rating"] = pd.to_numeric(
    df["rating"], errors="coerce"
)

df["availability"] = (
    df["availability"].astype(str).str.lower()
    .map({"true": True, "false": False})
)

df.dropna(
    subset=[
        "title", "category", "price_gbp",
        "rating", "availability"
    ],
    inplace=True
)

df = df[
    df["rating"].between(1, 5) &
    (df["price_gbp"] >= 0)
]

df["rating"] = df["rating"].astype(int)
df["price_inr"] = (
    df["price_gbp"] * GBP_TO_INR
).round(2)

df.to_csv("cleaned_books.csv", index=False)

print("Cleaned rows:", len(df))
display(df.head())

Cleaned rows: 69


,title,category,price_gbp,rating,availability,book_url,price_inr
0,It's Only the Himalayas,Travel,45.17,2,True,https://books.toscrape.com/catalogue/its-only-...,4765.44
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Travel,49.43,4,True,https://books.toscrape.com/catalogue/full-moon...,5214.86
2,See America: A Celebration of Our National Par...,Travel,48.87,3,True,https://books.toscrape.com/catalogue/see-ameri...,5155.78
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,2,True,https://books.toscrape.com/catalogue/vagabondi...,3897.17
4,Under the Tuscan Sun,Travel,37.33,3,True,https://books.toscrape.com/catalogue/under-the...,3938.31


In [5]:
import sqlite3

conn = sqlite3.connect("books.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS Categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    category_id INTEGER,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    availability BOOLEAN,
    book_url TEXT,
    FOREIGN KEY (category_id)
        REFERENCES Categories(category_id)
)
""")

# Clear tables before inserting fresh data
cursor.execute("DELETE FROM Books")
cursor.execute("DELETE FROM Categories")

for category in df["category"].unique():
    cursor.execute(
        "INSERT INTO Categories (category_name) VALUES (?)",
        (category,)
    )

cursor.execute("""
SELECT category_id, category_name FROM Categories
""")

category_map = {
    name: category_id
    for category_id, name in cursor.fetchall()
}

for _, row in df.iterrows():
    cursor.execute("""
    INSERT INTO Books (
        title, category_id, price_gbp,
        price_inr, rating, availability, book_url
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        category_map[row["category"]],
        row["price_gbp"],
        row["price_inr"],
        int(row["rating"]),
        int(row["availability"]),
        row["book_url"]
    ))

conn.commit()

print("Books:", cursor.execute(
    "SELECT COUNT(*) FROM Books"
).fetchone()[0])

print("Categories:", cursor.execute(
    "SELECT COUNT(*) FROM Categories"
).fetchone()[0])

Books: 69
Categories: 3


In [6]:
sql_result = pd.read_sql_query("""
SELECT
    c.category_name,
    COUNT(b.book_id) AS total_books,
    ROUND(AVG(b.price_inr), 2) AS average_price_inr
FROM Categories c
JOIN Books b
    ON c.category_id = b.category_id
GROUP BY c.category_name
ORDER BY average_price_inr DESC
""", conn)

display(sql_result)

# Reproduce the same result using pandas
pandas_result = (
    df.groupby("category")
    .agg(
        total_books=("title", "count"),
        average_price_inr=("price_inr", "mean")
    )
    .reset_index()
    .rename(columns={"category": "category_name"})
)

pandas_result["average_price_inr"] = (
    pandas_result["average_price_inr"].round(2)
)

pandas_result = pandas_result.sort_values(
    "average_price_inr", ascending=False
).reset_index(drop=True)

display(pandas_result)

print("SQL and pandas results:")
print(
    sql_result.equals(pandas_result)
)

,category_name,total_books,average_price_inr
0,Travel,11,4198.32
1,Historical Fiction,26,3549.47
2,Mystery,32,3346.36


,category_name,total_books,average_price_inr
0,Travel,11,4198.32
1,Historical Fiction,26,3549.47
2,Mystery,32,3346.36


SQL and pandas results:
True
